### General settings

In [1]:
mail = "bianchigianpaolo2@gmail.com"
user_name = "Pastasciutta"

from google.colab import drive
import os
import sys

drive.mount('/content/gdrive/')

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/Colab Notebooks'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

#Import client
from millionaire_client import MillionaireClient, AuthenticationError

#Get password
from google.colab import userdata
pwd = userdata.get('poli-millionaire')

#Login
API_URL = "http://131.175.15.22:51111/"
username = user_name
password = pwd
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).
['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/Colab Notebooks']

Welcome, Pastasciutta! (Role: student)


In [2]:
comp_id = 1 #entertainment, history, science, math (0-3)

def play_game(game):
  # Play the game
  time_to_answer = 0
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      for opt in question.options:
        print(f"{opt.id}: {opt.text}")
      print()

      option_texts = [opt.text for opt in question.options]
      response_id, response_text = pick_answer(tokenizer, model, question.text, option_texts)
      print(f"Selected answer: {response_id} - {response_text}")

      time_to_answer += 30 - game.time_remaining
      result = game.answer(response_id)
      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
        return game.current_level, -1
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")
  avg_response_time = time_to_answer / game.current_level
  return game.current_level, avg_response_time


# RAG model

First, we will set the model and its corresponding prompts.



In [3]:
!pip install -U transformers accelerate bitsandbytes
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util
import torch

#quantize to 4 bits to make it "smaller"
quant_config = BitsAndBytesConfig(load_in_4bit=True)

#load model and tokenizer
model_id = "Qwen/Qwen2.5-14B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

#chunk embedder
embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Functions

In [4]:
import requests
import re
import numpy as np
from concurrent.futures import ThreadPoolExecutor

#Prompt to get best queries given the question and options from the model
#Note that it generates between 1 and 5 queries, and a canonical example has been provided
def build_query_prompt(question, options):
    system_prompt = {"role": "system", "content": """#TASK: You are helping answer a Who Wants to Be a Millionaire question by generating Wikipedia search queries.

#RULES:
- Generate between 1 and 5 queries focused on the TOPIC and CONTEXT of the question, not the answer options.
- Never copy the options verbatim as queries — they are rarely good search terms.
- Think about what Wikipedia articles would contain the answer and search for those.
- Each query on a new line.
- No numbering, no bullets, no explanations, no quotes.
- Output only the queries.

#EXAMPLE:
Question: What term describes the founding of Carthage by the legendary Queen Dido?
Good queries:
Queen Dido founding of Carthage
Carthage history origin Phoenician
Dido legendary queen Carthage"""}

    user_prompt = {"role": "user", "content": f"""#QUESTION: {question}

#OPTIONS:
A) {options[0]}
B) {options[1]}
C) {options[2]}
D) {options[3]}"""}

    return [system_prompt, user_prompt]

#Prompt to get the answer given question, options and documents
def build_doc_prompt(question, options, documents):
    system_prompt = {"role": "system", "content": """#TASK:
You are an expert quiz player competing on Who Wants to Be a Millionaire.
You will receive a question, four labeled options (A, B, C, D), and a set of reference documents retrieved from Wikipedia.
Use the documents as your primary source of truth to determine the correct answer.

#RULES:
Respond with ONLY the letter of the correct answer: A, B, C, or D.
Before answering, think step by step about what the documents say. Then output only the letter.
Do not explain your reasoning. Do not write anything else."""}

    formatted_docs = "\n\n".join(f"[Document {i+1}]\n{doc}" for i, doc in enumerate(documents))

    user_prompt = {"role": "user", "content": f"""#QUESTION: {question}
#OPTIONS:
A) {options[0]}
B) {options[1]}
C) {options[2]}
D) {options[3]}

#REFERENCE DOCUMENTS:
{formatted_docs}"""}

    return [system_prompt, user_prompt]

#Given a text, it divides it into chunks of N words
def chunk_text(text, chunk_size=200):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

#Can be rewritten more cleanly using util_cos_sim like we did in the lectures (from sentence-tr utils)
def get_top_chunks(question, options, chunks, top_k=3):
    # Embed question and each option separately
    question_embedding = embedder.encode(question)
    option_embeddings = embedder.encode(options)
    chunk_embeddings = embedder.encode(chunks)

    # Normalize everything
    chunks_norm = chunk_embeddings / np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)
    question_norm = question_embedding / np.linalg.norm(question_embedding)
    options_norm = option_embeddings / np.linalg.norm(option_embeddings, axis=1, keepdims=True)

    # Similarity with question
    question_sim = chunks_norm @ question_norm

    # Max similarity across all options (not average — max preserves signal)
    option_sims = chunks_norm @ options_norm.T  # shape: (n_chunks, 4)
    best_option_sim = option_sims.max(axis=1)   # best matching option per chunk

    # Combine: weight question more heavily than options
    combined = 0.6 * question_sim + 0.4 * best_option_sim

    top_indices = np.argsort(combined)[::-1][:top_k]
    return [chunks[i] for i in top_indices]

#Multiple parallel queries to Wikipedia API
def wikisearch_multi(queries, question, options):
    def search_one(query):
        query = query.strip().replace('"', '')
        if not query:
            return []
        return wikisearch_single(query)

    with ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(search_one, queries))

    #Pool all chunks from all queries
    all_chunks = [chunk for doc_chunks in results for chunk in doc_chunks]

    if not all_chunks:
        return []

    return get_top_chunks(question, options, all_chunks, top_k=3)

HEADERS = {
    "User-Agent": f"WhoWantsToBeAMillionaire-Bot/1.0 (research project; {mail})"
}

#Single wikipedia call
def wikisearch_single(query):
    search_url = "https://en.wikipedia.org/w/api.php"
    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": 3
    }
    search_response = requests.get(search_url, params=search_params, headers=HEADERS)

    #No titles or rate limited, a log could be made here
    if not search_response.text or search_response.status_code != 200:
        return []

    titles = [r["title"] for r in search_response.json()["query"]["search"]]

    all_chunks = []
    for title in titles:
        extract_params = {
            "action": "query",
            "titles": title,
            "prop": "extracts",
            "explaintext": True,
            "format": "json"
        }
        extract_response = requests.get(search_url, params=extract_params, headers=HEADERS)

        #No documents or rate limited, a log could be made here
        if not extract_response.text or extract_response.status_code != 200:
            continue

        pages = extract_response.json()["query"]["pages"]
        for page in pages.values():
            if "extract" in page:
                all_chunks.extend(chunk_text(page["extract"]))

    return all_chunks

#General pipeline
def pick_answer(tokenizer, model, question, options):
    #Step 1: generate multiple search queries
    model_prompt = build_query_prompt(question, options)
    print("Generating search queries...")
    raw_queries = call_model(tokenizer, model, model_prompt).strip()
    queries = [q.strip() for q in raw_queries.split("\n") if q.strip()]
    print(f"Queries: {queries}")

    #Step 2: fetch in parallel, pool and rank chunks
    found_docs = wikisearch_multi(queries, question, options)
    print(f"Retrieved {len(found_docs)} chunks")

    #Step 3: answer with documents
    model_prompt = build_doc_prompt(question, options, found_docs)
    print("Answering with documents...")
    response = call_model(tokenizer, model, model_prompt)
    print(f"Model answered: {response}")

    letter_to_index = {"A": 0, "B": 1, "C": 2, "D": 3}
    match = re.search(r"\b(A|B|C|D)\b", response)
    if match:
        letter = match.group(1)
        return letter_to_index[letter], options[letter_to_index[letter]]
    else:
        print("Model did not output valid response")
        return None, None

#General call to LLM model
def call_model(tokenizer, model, prompt):
    inputs = tokenizer.apply_chat_template(
        prompt,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True
    ).to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        pad_token_id=tokenizer.eos_token_id
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response

# Run game

In [5]:
#Play game
def game_start(comp_id):
    print("\n=== Starting Game ===")
    game = client.game.start(competition_id=comp_id)
    print(f"Session ID: {game.session_id}")
    print(f"Total number of questions: {game.state.competition.max_levels}")
    print()
    return play_game(game)

all_stats = []
for comp_id in range(0, 4):
    for run in range(1, 4):
        print(f"\n>>> Competition {comp_id} | Run {run}")
        level_reached, avg_response_time = game_start(comp_id)
        all_stats.append({
            "model": model_id,
            "comp_id": comp_id,
            "run": run,
            "level_reached": level_reached,
            "avg_response_time": avg_response_time
        })

#Print summary
import pandas as pd
df = pd.DataFrame(all_stats)
print(df)
df.to_csv("results.csv", index=False)


>>> Competition 0 | Run 1

=== Starting Game ===
Session ID: 45978
Total number of questions: 15


--- Level 1 ---
Q: How does Al Pacino's role as Michael Corleone in The Godfather differ from his role as Tony Montana in Scarface in terms of character development?
0: Michael Corleone is a young up-and-comer, while Tony Montana is an older, wiser leader.
1: Michael Corleone is a street punk, while Tony Montana is a high-level mafia boss.
2: Michael Corleone is a rising criminal, while Tony Montana is a fall from grace.
3: Michael Corleone is a quiet and calculating leader, while Tony Montana is more impulsive and violent.

Generating search queries...
Queries: ['Al Pacino Michael Corleone character development', 'Al Pacino Tony Montana character differences', 'Michael Corleone vs Tony Montana transformation', 'Godfather Michael Corleone leadership style', 'Scarface Tony Montana personality traits']
Retrieved 3 chunks
Answering with documents...
Model answered: D
Selected answer: 3 - Mi

Are we serious?

The question was "Q: What is the fundamental principle behind the naming of Athens according to modern scholars?

0: The name was derived from the first olive tree planted in Athens.

1: The name reflects the city's patronage by Athena, the goddess of wisdom.

2: The goddess Athena took her name from the city.

3: The name comes from the word 'flower' or 'flowering city'."

Wikipedia extract: "According to Greek mythology, the city was named after Athena, the ancient Greek goddess of wisdom, but modern scholars generally agree that the goddess took her name after the city". But 2 is incorrect according to the game

Notes:

Try changing model size (prompts could also be changed but do not affect memory consumption)

Try different categories

Try thinking token (time out possibility)

Try different embedding algorithms

Write statistics